# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Sheheryar-khan/FlyRank-ML-Internship-Starter-Repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
%pip -q install duckdb
import os
import duckdb
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

In [4]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_h1,
        SUM(gsc_clicks) AS clicks_h1,
        AVG(gsc_avg_position) AS avg_position_h1,
        COUNT(DISTINCT report_date) AS active_days_h1
    FROM {MAR}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY content_hash_id
    HAVING impressions_h1 >= 20
""").df()

features["ctr_h1"] = features["clicks_h1"] / features["impressions_h1"] * 100

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(109592, 6)


,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,active_days_h1,ctr_h1
0,content_b7e512995f79d5a6,429.0,2.0,4.247255,15,0.466200
1,content_905aa32a0230694e,89.0,0.0,3.010741,15,0.000000
2,content_05434271b257bb68,628.0,1.0,5.330069,15,0.159236
3,content_d056587ff7faca0c,1280.0,9.0,4.468441,15,0.703125
4,content_2662845f598544ef,97.0,0.0,8.765983,15,0.000000


In [5]:
labels = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_h2
    FROM {MAR}
    WHERE report_date > DATE '2026-03-15'
    GROUP BY content_hash_id
""").df()

data = features.merge(labels, on="content_hash_id", how="left")
data["impressions_h2"] = data["impressions_h2"].fillna(0)

data["is_declining"] = (data["impressions_h2"] < 0.8 * data["impressions_h1"]).astype(int)

print("declining rate:", data["is_declining"].mean().round(3))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

declining rate: 0.291


,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,active_days_h1,ctr_h1,impressions_h2,is_declining
0,content_b7e512995f79d5a6,429.0,2.0,4.247255,15,0.466200,711.0,0
1,content_905aa32a0230694e,89.0,0.0,3.010741,15,0.000000,60.0,1
2,content_05434271b257bb68,628.0,1.0,5.330069,15,0.159236,793.0,0
3,content_d056587ff7faca0c,1280.0,9.0,4.468441,15,0.703125,1490.0,0
4,content_2662845f598544ef,97.0,0.0,8.765983,15,0.000000,53.0,1


In [11]:
dim_content = con.sql(f"""
    SELECT content_hash_id, content_type, content_created_date, content_updated_date
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()

print(dim_content.shape)
dim_content.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(519606, 4)


,content_hash_id,content_type,content_created_date,content_updated_date
0,content_004de9653278b5a4,keyword article,2026-05-30,2026-07-01
1,content_00dc5efae381b2ab,keyword article,2026-06-12,2026-07-01
2,content_01410f2556c327ac,keyword article,2026-05-09,2026-07-01
3,content_019f27f634053ca7,keyword article,2026-06-15,2026-06-15
4,content_01efa71faea45dcc,keyword article,2026-05-21,2026-06-01


In [12]:
client_lookup = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM {MAR}
""").df()

data = data.merge(client_lookup, on="content_hash_id", how="left")
data.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(109592, 9)

## 1. My rule and its reason codes

My rule: flag a page if it's old AND still gets real traffic. I'm checking two things first — does "old" actually predict decline, and does CTR really drop as position gets worse — before I build the rule on them.

Reason code: stale_visible_page.

In [14]:
# check dim_content actually has it
print(dim_content.columns.tolist())

# merge it onto data
data = data.merge(dim_content[["content_hash_id", "content_updated_date"]],
                   on="content_hash_id", how="left")

print(data.columns.tolist())

['content_hash_id', 'content_type', 'content_created_date', 'content_updated_date']
['content_hash_id', 'impressions_h1', 'clicks_h1', 'avg_position_h1', 'active_days_h1', 'ctr_h1', 'impressions_h2', 'is_declining', 'client_hash_id', 'content_updated_date']


In [15]:
data["content_updated_date"] = pd.to_datetime(data["content_updated_date"])

data["days_since_update"] = (pd.Timestamp("2026-03-15") - data["content_updated_date"]).dt.days
data["staleness_tier"] = pd.cut(data["days_since_update"], bins=[0,90,180,365,99999],
                                 labels=["0-90","90-180","180-365","365+"])

bucket1 = data.groupby("staleness_tier").agg(
    n=("content_hash_id", "size"),
    decline_rate=("is_declining", "mean")
).round(3)
print(bucket1)

                    n  decline_rate
staleness_tier                     
0-90            20239         0.371
90-180            176         0.335
180-365            26         0.385
365+                0           NaN


/tmp/ipykernel_1454/456571133.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = data.groupby("staleness_tier").agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.